In [ ]:
from tqdm import tqdm
from datetime import datetime, timezone
import pandas as pd
import os, warnings
from pandas.errors import ParserWarning
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
warnings.simplefilter('ignore', ParserWarning)
warnings.simplefilter('ignore', UserWarning)
import plotly.io as pio
import seaborn as sns
from sklearn.preprocessing import StandardScaler

'''LABEL_MAP = {
    9: "prefer warmer",
    11: "prefer cooler",  
    10: "no change"
}'''

# COLOR_MAP = {
#     "prefer warmer": "red",
#     "prefer cooler": "blue",
#     "no change": "green"
# }

# COLOR_MAP = {
#     "prefer warmer": "#EF6A72",  # muted brick red
#     "no change":     "#B0BEC5",  # sage/forest green
#     "prefer cooler": "#81AFE4",  # slate/steel blue
# }

import matplotlib.colors as mcolors
import colorsys

def darken(hex_color, factor=0.7):
    # factor < 1 darkens, factor > 1 lightens
    r, g, b = mcolors.to_rgb(hex_color)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    l = max(0, min(1, l * factor))
    r, g, b = colorsys.hls_to_rgb(h, l, s)
    return mcolors.to_hex((r, g, b))

COLOR_MAP = {
    "prefer warmer": darken("#EF6A72", 0.75),
    "no change":     darken("#B0BEC5", 0.6),
    "prefer cooler": darken("#81AFE4", 0.75),
}

LABEL_MAP = {
    "Wärmer": "prefer warmer",
    "Kühler": "prefer cooler",
    "Keine Veränderung": "no change"
}

os.makedirs("../figures/pca", exist_ok=True)
os.makedirs("../figures/correlation", exist_ok=True)


#from PersonAI.anchor import data_dir

# Get Data


In [ ]:
# data path
#DATA_PATH = data_dir()

df = pd.read_csv("../../data/personai/all_aircontrol_with_surveys_hr_with_new_bpm_new.csv")
df["q_preference_temperature"] = df["q_preference_temperature"].map(LABEL_MAP)
df.head()

In [ ]:
df_temp = df[['user_id', 'q_preference_temperature']].groupby(['user_id', 'q_preference_temperature']).agg('size').reset_index(name='count').pivot(index='user_id', columns='q_preference_temperature', values='count')
df_temp_perc = (df_temp.div(df_temp.sum(axis=1), axis=0) * 100).round(0).reset_index()
df_temp = df_temp.reset_index()
df_temp.columns.name,df_temp_perc.columns.name =None,None
df_temp.head()

# Histogram of temperature preference per participant



In [ ]:
cols = ["no change", "prefer cooler", "prefer warmer"]
df_temp_perc.sort_values(cols).plot(
    x = 'user_id',
    kind = 'barh',
    stacked = True,
    title = 'Participants Temperature Preference',
    color=[COLOR_MAP[c] for c in cols],
    mark_right = True,
    figsize=(12,18))



# Feature Distribution across preferences

In [ ]:

df_num = df.loc[:,["user_id","q_preference_temperature","AirTemp","Humidity","CO2","bpm"]]

scaler = StandardScaler()
df_num[["AirTemp","Humidity","CO2","bpm"]] = scaler.fit_transform(df_num[["AirTemp","Humidity","CO2","bpm"]])

df_num_stack = df_num.set_index(['user_id','q_preference_temperature']).stack().reset_index()

df_num_stack.columns = ["user_id","preference_temperature","numerical_feature","value"]


In [ ]:
chunks = [df.user_id.unique()[i:i + 8] for i in range(0, len(df.user_id.unique()), 9)]

chunk = chunks[0]

for chunk in chunks:
    print(chunk)
    #sns.FacetGrid(df_num_stack, col="user_id")
    fig, axes = plt.subplots(3, 3, figsize=(18, 10))

    for i in range(3):
            for j in range(3):
                if len(chunk)>i+j:
                    c = chunk[i+j]
                    sns.swarmplot(
                        ax=axes[i, j],
                        data=df_num_stack.loc[df_num_stack.user_id == c],
                        x="value",
                        y="numerical_feature",
                        hue="preference_temperature",
                        palette=COLOR_MAP
                    ).set_title(c)

                    #y="q_preference_temperature")#, hue="sex")



In [ ]:
#sns.swarmplot(data=df_num_stack, x="value",y = "numerical_feature",palette=COLOR_MAP, hue = "preference_temperature")#y="q_preference_temperature")#, hue="sex")
# sns.swarmplot(ax=axes[0, 0],data=df_num.loc[df_num.user_id == "id001"], x="value",y = "numerical_feature", hue = "preference_temperature")#y="q_preference_temperature")#, hue="sex")

sns.stripplot(
    data=df_num_stack,
    x="value",
    y="numerical_feature",
    hue="preference_temperature",
    palette=COLOR_MAP,
    jitter=True
)

# Feature Distribution

In [ ]:
# sns.set()

fig, ax = plt.subplots(2,2,figsize=(10,4))

sns.kdeplot(ax=ax[0,0], data=df,
            x="AirTemp", palette=COLOR_MAP, hue="q_preference_temperature" ,
            bw_adjust=0.3,
            linewidth=2.5, fill=True)
sns.kdeplot(ax=ax[0,1], data=df,
            x="Humidity", palette=COLOR_MAP, hue="q_preference_temperature" ,
            bw_adjust=0.3,
            linewidth=2.5, fill=True)
sns.kdeplot(ax=ax[1,0], data=df,
            x="CO2", palette=COLOR_MAP, hue="q_preference_temperature" ,
            bw_adjust=0.3,
            linewidth=2.5, fill=True)
sns.kdeplot(ax=ax[1,1], data=df,
            x="bpm", palette=COLOR_MAP, hue="q_preference_temperature" ,
            bw_adjust=0.3,
            linewidth=2.5, fill=True)

In [ ]:
# sns.set()

fig, ax = plt.subplots(2,2,figsize=(10,4))

sns.kdeplot(ax=ax[0,0], data=df_num,
            x="AirTemp", palette=COLOR_MAP, hue="q_preference_temperature" ,
            bw_adjust=0.3,
            linewidth=2.5, fill=True)
sns.kdeplot(ax=ax[0,1], data=df_num,
            x="Humidity", palette=COLOR_MAP, hue="q_preference_temperature" ,
            bw_adjust=0.3,
            linewidth=2.5, fill=True)
sns.kdeplot(ax=ax[1,0], data=df_num,
            x="CO2", palette=COLOR_MAP, hue="q_preference_temperature" ,
            bw_adjust=0.3,
            linewidth=2.5, fill=True)
sns.kdeplot(ax=ax[1,1], data=df_num,
            x="bpm", palette=COLOR_MAP, hue="q_preference_temperature" ,
            bw_adjust=0.3,
            linewidth=2.5, fill=True)


In [ ]:

fig, ax = plt.subplots(2,2,figsize=(10,4))

sns.histplot(ax=ax[0,0], data=df_num, x="AirTemp",palette=COLOR_MAP, hue="q_preference_temperature",kde=True,bins=30)
sns.histplot(ax=ax[0,1], data=df_num, x="AirTemp", palette=COLOR_MAP, hue="q_preference_temperature",kde=True,bins=50)
sns.histplot(ax=ax[1,0], data=df_num, x="AirTemp", palette=COLOR_MAP, hue="q_preference_temperature",kde=True,bins=100)
sns.histplot(ax=ax[1,1], data=df_num, x="AirTemp", palette=COLOR_MAP, hue="q_preference_temperature",kde=True,bins=200)


In [ ]:

fig, ax = plt.subplots(2,2,figsize=(10,4))

sns.histplot(ax=ax[0,0], data=df_num, x="Humidity", palette=COLOR_MAP, hue="q_preference_temperature",kde=True,bins=30)
sns.histplot(ax=ax[0,1], data=df_num, x="Humidity", palette=COLOR_MAP, hue="q_preference_temperature",kde=True,bins=50)
sns.histplot(ax=ax[1,0], data=df_num, x="Humidity", palette=COLOR_MAP, hue="q_preference_temperature",kde=True,bins=100)
sns.histplot(ax=ax[1,1], data=df_num, x="Humidity", palette=COLOR_MAP, hue="q_preference_temperature",kde=True,bins=200)


In [ ]:
fig, ax = plt.subplots(2,2,figsize=(10,4))

sns.histplot(ax=ax[0,0], data=df_num, x="CO2", palette=COLOR_MAP, hue="q_preference_temperature",kde=True,bins=30)
sns.histplot(ax=ax[0,1], data=df_num, x="CO2", palette=COLOR_MAP, hue="q_preference_temperature",kde=True,bins=50)
sns.histplot(ax=ax[1,0], data=df_num, x="CO2", palette=COLOR_MAP, hue="q_preference_temperature",kde=True,bins=100)
sns.histplot(ax=ax[1,1], data=df_num, x="CO2", palette=COLOR_MAP, hue="q_preference_temperature",kde=True,bins=200)


In [ ]:
fig, ax = plt.subplots(2,2,figsize=(10,4))

sns.histplot(ax=ax[0,0], data=df_num, x="bpm", palette=COLOR_MAP, hue="q_preference_temperature",kde=True,bins=30)
sns.histplot(ax=ax[0,1], data=df_num, x="bpm", palette=COLOR_MAP, hue="q_preference_temperature",kde=True,bins=50)
sns.histplot(ax=ax[1,0], data=df_num, x="bpm", palette=COLOR_MAP, hue="q_preference_temperature",kde=True,bins=100)
sns.histplot(ax=ax[1,1], data=df_num, x="bpm", palette=COLOR_MAP, hue="q_preference_temperature",kde=True,bins=200)

In [ ]:
fig, ax = plt.subplots(2,2,figsize=(10,4))

sns.histplot(ax=ax[0,0], data=df_num, x="AirTemp", palette=COLOR_MAP, hue="q_preference_temperature",kde=True,bins=50)
sns.histplot(ax=ax[0,1], data=df_num, x="Humidity", palette=COLOR_MAP, hue="q_preference_temperature",kde=True,bins=50)
sns.histplot(ax=ax[1,0], data=df_num, x="CO2", palette=COLOR_MAP, hue="q_preference_temperature",kde=True,bins=50)
sns.histplot(ax=ax[1,1], data=df_num, x="bpm", palette=COLOR_MAP, hue="q_preference_temperature",kde=True,bins=50)

In [ ]:
sns.pairplot(df.loc[:,[ 'AirTemp','Humidity', 'CO2','bpm',]])

In [ ]:
sns.pairplot(df_num.loc[:,[ 'AirTemp','Humidity', 'CO2','bpm',]])

# Basic statistics
Make more sense on the original data then the already cleaned one

In [ ]:
print(df.dtypes)
print(df.describe())

In [ ]:
df.loc[:,['AirTemp','Humidity', 'CO2','bpm']].quantile([0,0.05,0.1,0.25, 0.5,0.75,0.9,1])
#CO2 & bpm, maybe some outliers => other standartization

In [ ]:
df.isna().sum()

In [ ]:
df[df.duplicated()]

# Correlation

and other measrurements of influence


In [ ]:
# chi2_contingency(pd.crosstab(df['q_preference_temperature'],df['q_activity']))
from scipy.stats import chi2_contingency
def ch_calculate(df):
    factors_paired = [(i,j) for i in df.columns.values for j in df.columns.values] 

    chi2, p_values =[], []

    for f in factors_paired:
        if f[0] != f[1]:
            chitest = chi2_contingency(pd.crosstab(df[f[0]], df[f[1]]))   
            chi2.append(chitest[0])
            p_values.append(chitest[1])
        else:      # for same factor pair
            chi2.append(0)
            p_values.append(0)

    chi2 = np.array(chi2).reshape((len(df.columns),len(df.columns))) # shape it as a matrix
    chi2 = pd.DataFrame(chi2, index=df.columns.values, columns=df.columns.values) # then a df for convenience
    fig, ax = plt.subplots(figsize=(30,30))
    sns.heatmap(chi2, annot = True)
    #plt.title("Chi2 values for each pair of factors")
    plt.savefig("../figures/correlation/correlation_chi2_heatmap.pdf")
    plt.show()
    return(chi2)

   
ch_calculate(df.loc[:,['q_preference_temperature','user_id','q_activity','q_feeling','q_heartbeat', 'bpm']]) #bpm_method

In [ ]:
# import pandas as pd
# from scipy.stats import pearsonr

# def pearsonr_pval(x, y):
#     return pearsonr(x, y)[1]


# df = (
#     pd.concat(
#         [
#             x.corr(method="pearson").reset_index().assign(value="r"),
#             x.corr(method=pearsonr_pval).reset_index().assign(value="p"),
#         ]
#     )
#     .groupby(["index", "value"])
#     .agg(lambda x: list(x)[0])
# ).sort_index(ascending=[True, False])

# df.index.names = ["", ""]
df.loc[:,['AirTemp','Humidity','CO2','bpm']].corr()

In [ ]:
from scipy.stats import chi2_contingency
def ch_calculate(df):
    factors_paired = [(i,j) for i in df.columns.values for j in df.columns.values] 

    chi2, p_values =[], []

    for f in factors_paired:
        if f[0] != f[1]:
            chitest = chi2_contingency(pd.crosstab(df[f[0]], df[f[1]]))   
            chi2.append(chitest[0])
            p_values.append(chitest[1])
        else:      # for same factor pair
            chi2.append(0)
            p_values.append(0)

    chi2 = np.array(chi2).reshape((len(df.columns),len(df.columns))) # shape it as a matrix
    chi2 = pd.DataFrame(chi2, index=df.columns.values, columns=df.columns.values) # then a df for convenience
    fig, ax = plt.subplots(figsize=(30,30))
    sns.heatmap(chi2, annot = True)
    plt.show()
    return(chi2)

   
ch_calculate(df.loc[:,['q_preference_temperature','user_id','q_activity','q_feeling','q_heartbeat','bpm']]) #bpm_method

In [ ]:
from scipy.stats import f_oneway
#statistic, pvalue = f_oneway(df['AirTemp'],df['q_preference_temperature'])

groups = [
    group["AirTemp"].values
    for _, group in df.groupby("q_preference_temperature")
]

statistic, pvalue = f_oneway(*groups)

### Correlation(extra)

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
import unicodedata, re
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import umap.umap_ as umap
from sklearn.manifold import TSNE
from mpl_toolkits.mplot3d import Axes3D
import sys, pathlib
#sys.path.append("..")
#from anchor import data_dir

In [ ]:
# correlation ratio (categorical-numeric)
def eta_corr(df, target_col, feature_cols):
    
    def eta(y, x):
        d = pd.DataFrame({"y": y, "x": pd.to_numeric(x, errors="coerce")}).dropna()
        if d["y"].nunique() < 2 or d["x"].nunique() < 2: return np.nan
        m = d["x"].mean()
        ssb = d.groupby("y")["x"].apply(lambda g: len(g)*(g.mean()-m)**2).sum()
        sst = ((d["x"]-m)**2).sum()
        return float(np.sqrt(ssb/sst)) if sst > 0 else np.nan
    
    y = df[target_col]
    vals = {col: eta(y, df[col]) for col in feature_cols}
    return pd.DataFrame.from_dict(vals, orient="index", columns=["eta"]).sort_values("eta", ascending=False)

# correlation (numeric-numeric)
def num_corr(df, cols, method="pearson"):
    X = df[cols].apply(pd.to_numeric, errors="coerce")
    return X.corr(method=method)

# correlation (categorical-categorical)
def cramers_v_corr(df, cols): 
    k = len(cols) 
    V = np.ones((k, k)) 
    for i in range(k): 
        for j in range(k): 
            tab = pd.crosstab(df[cols[i]], df[cols[j]]) 
            chi2 = chi2_contingency(tab)[0] 
            n = tab.values.sum() 
            r, c = tab.shape 
            m = min(r, c) - 1
            V[i, j] = np.sqrt(chi2 / (n * m)) if n > 0 and m > 0 else np.nan
    return pd.DataFrame(V, index=cols, columns=cols)

### dataset setup

In [ ]:
def build_dfm(main_path, onb_path,id_main="user_id", id_onb="id",target_col="q_preference_temperature"):
    df  = pd.read_csv(main_path)
    onb = pd.read_csv(onb_path)
    df.columns  = df.columns.str.replace(r"\s+", " ", regex=True).str.strip()
    onb.columns = onb.columns.str.replace(r"\s+", " ", regex=True).str.strip()
    df[id_main]  = df[id_main].astype(str).str.strip()
    onb[id_onb]  = onb[id_onb].astype(str).str.strip()
    onb = onb.sort_values(id_onb).drop_duplicates(subset=[id_onb], keep="first")
    dfm = df.merge(onb, left_on=id_main, right_on=id_onb, how="left")

    def clean_de(s):
        s = str(s).lower()
        s = unicodedata.normalize("NFKD", s).encode("ascii","ignore").decode("utf-8")  # kühler->kuhler
        s = re.sub(r"[-_]+"," ", s)
        s = re.sub(r"\s+"," ", s).strip()
        return s
    
    dfm["target_clean"] = dfm[target_col].map(clean_de).replace({"keineveranderung":"keine verand erung"})
    return dfm

def one_hot_encode(dfm, cat_cols):
    cat_cols = [c for c in cat_cols if c in dfm.columns]
    df_only_enc = pd.get_dummies(dfm[cat_cols], drop_first=True, dtype=np.uint8)
    return df_only_enc

#DATA_PATH = data_dir()
#main_path = os.path.join(DATA_PATH,"all_aircontrol_with_surveys_hr_FIXED.csv")
#onb_path  = os.path.join(DATA_PATH, "onboarding_survey.csv")
main_path = "../../data/personai/all_aircontrol_with_surveys_hr_with_new_bpm_new.csv"
onb_path = "../../data/personai/onboarding_survey.csv"

dfm = build_dfm(main_path, onb_path)
dfm["q_preference_temperature"] = dfm["q_preference_temperature"].map(LABEL_MAP)
print(dfm["q_preference_temperature"])

### categorical features correlation (one-hot encoded)

In [ ]:
cat_cols = ["q_location", "q_activity", "q_feeling", "q_heartbeat", "bpm_method", "Geschlecht:", "Höchster Bildungsabschluss:","Haben Sie mehrere Jahre (5+) außerhalb von Österreich gelebt?", 
            "Art der Wohnimmobilie:", "Baualter der Immobilie", "Klimatisierungsmöglichkeit:", "Heizungssystem:", "Wurde die Immobilie in den letzten 20 Jahren saniert?", 
            "Wie würden Sie das durchschnittliche Raumklima in Ihrem Zuhause einschätzen? Unter Raumklima verstehen wir Faktoren wie Temperatur, Luftfeuchtigkeit, Lichtverhältnisse, Lärmpegel, Luftfrische usw.", 
            "Berufliche Tätigkeit (vorwiegend):", "Geben Sie bitte die Räumlichkeiten in Ihrem Büro an:", "Klimatisierungsmöglichkeit im Büro:", "Heizungssystem im Büro:", 
            "Wie würden Sie das durchschnittliche Raumklima in Ihrem Büro einschätzen? Unter Raumklima verstehen wir Faktoren wie Temperatur, Luftfeuchtigkeit, Lichtverhältnisse, Lärmpegel, Luftfrische usw.", 
            "Wie schätzen Sie Ihr Fitnesslevel ein?"]
df_only_enc = one_hot_encode(dfm, cat_cols)
print(df_only_enc.shape)

In [ ]:
#mapping = {"Kühler": -1, "Keine Veränderung": 0, "Wärmer": 1}
mapping = {"prefer warmer": -1, "prefer no change": 0, "prefer cooler": 1}
df_only_enc["target_num"] = dfm["q_preference_temperature"].map(mapping).astype(float)

X = df_only_enc.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan).dropna(axis=0)

# correlate each feature with target_num, then sort by |corr|
corr_vs_target = X.corrwith(X["target_num"]).drop(labels=["target_num"])
corr_vs_target = corr_vs_target.reindex(corr_vs_target.abs().sort_values(ascending=False).index)

# small, readable heatmap
plt.figure(figsize=(6, max(2, len(corr_vs_target)//2)), dpi=130)
sns.heatmap(corr_vs_target.to_frame("target_num"), annot=True)
#plt.title("Pearson correlation: categorical features one-hot encoded")
plt.savefig("../figures/correlation/correlation_pearson_onehot_target.pdf")
plt.tight_layout()
plt.show()

### enviormental features correlation

In [ ]:
# using corelation ratio (eta)
if "CO2_log1p" not in dfm.columns:
    dfm["CO2_log1p"] = np.log1p(pd.to_numeric(dfm["CO2"], errors="coerce").clip(lower=0))

eta_env = eta_corr(dfm, "q_preference_temperature", ["AirTemp","Humidity","CO2","bpm","CO2_log1p"])
print(eta_env)

sns.heatmap(eta_env, annot=True)
#plt.title("Correlation ratio: enviormental features")
plt.tight_layout()
plt.savefig("../figures/correlation/correlation_eta_env.pdf")
plt.show()

In [ ]:
# encoding thermal preferences to (-1, 0, 1) and then using pearson correlation
if "CO2_log1p" not in dfm.columns:
    dfm["CO2_log1p"] = np.log1p(pd.to_numeric(dfm["CO2"], errors="coerce").clip(lower=0))

#mapping = {"Kühler": -1, "Keine Veränderung": 0, "Wärmer": 1}
mapping = {"prefer warmer": -1, "prefer no change": 0, "prefer cooler": 1}
dfm["target_num"] = dfm["q_preference_temperature"].map(mapping)

corr_env = num_corr(dfm, ["AirTemp","Humidity","CO2","bpm","CO2_log1p","target_num",], method="pearson")
print(corr_env)

sns.heatmap(corr_env, annot=True)
#plt.title("Pearson correlation: numeric features + target_num")
plt.tight_layout()
plt.savefig("../figures/correlation/correlation_pearson_numeric_target.pdf")
plt.show()

### time of day correlation

In [ ]:
# using corelation ratio (eta)
if not {"tod","sin_tod","cos_tod"}.issubset(dfm.columns):
    dt = pd.to_datetime(dfm["survey_time"], errors="coerce")
    dfm["tod"] = dt.dt.hour + dt.dt.minute/60 #converting every time to a numerical value(13:30 -> 13.5)
    ang = 2*np.pi*dfm["tod"]/24 #converting to angle in radians
    dfm["sin_tod"], dfm["cos_tod"] = np.sin(ang), np.cos(ang)  #cyclicality Analysis

eta_time = eta_corr(dfm, "target_clean", ["tod","sin_tod","cos_tod"])
print(eta_time)

sns.heatmap(eta_time, annot=True)
#plt.title("correlation ratio: time-of-day features")
plt.tight_layout()
plt.savefig("../figures/correlation/correlation_eta_tod.pdf")
plt.show()

In [ ]:
# encoding thermal preferences to (-1, 0, 1) and then use pearson correlation
#mapping = {"Kühler": -1, "Keine Veränderung": 0, "Wärmer": 1}
mapping = {"prefer warmer": -1, "prefer no change": 0, "prefer cooler": 1}
dfm["target_num"] = dfm["q_preference_temperature"].map(mapping)

if not {"tod", "sin_tod", "cos_tod"}.issubset(dfm.columns):
    dt = pd.to_datetime(dfm["survey_time"], errors="coerce")
    dfm["tod"] = dt.dt.hour + dt.dt.minute / 60 #converting every time to a numerical value (13:30 -> 13.5)
    ang = 2 * np.pi * dfm["tod"] / 24  #converting to angle in radians
    dfm["sin_tod"], dfm["cos_tod"] = np.sin(ang), np.cos(ang) #cyclicality Analysis

time_cols = ["tod", "sin_tod", "cos_tod", "target_num"]
corr_time = num_corr(dfm, time_cols, method="pearson")
print(corr_time)

sns.heatmap(corr_time, annot=True)
#plt.title("Pearson correlation: time-of-day features")
plt.tight_layout()
plt.savefig("../figures/correlation/correlation_pearson_tod_target.pdf")
plt.show()

### onboarding-scores correlation

In [ ]:
# using correlation ratio (eta)
score_cols = [c for c in dfm.columns if c.startswith(("BFI","BFI2S","BFI2XS","MAIA","BSI","SWLS"))]
eta_scores = eta_corr(dfm, "q_preference_temperature", score_cols)
print(eta_scores)

plt.figure(figsize=(7, 10))
sns.heatmap(eta_scores, annot=True)
#plt.title("correlation ratio: onboarding scores")
plt.tight_layout()
plt.savefig("../figures/correlation/correlation_eta_onboarding.pdf")
plt.show()

In [ ]:
# encoding thermal preferences to (-1, 0, 1) and then use pearson correlation
#map_clean = {"Kühler": -1, "Keine Veränderung": 0, "Wärmer": 1}
map_clean = {"prefer warmer": -1, "prefer no change": 0, "prefer cooler": 1}
dfm["target_num"] = dfm["q_preference_temperature"].map(map_clean)

score_cols = [c for c in dfm.columns if c.startswith(("BFI", "BFI2S", "BFI2XS", "MAIA", "BSI", "SWLS"))]
score_cols = [c for c in score_cols if c in dfm.columns]

corr_onb = num_corr(dfm, score_cols + ["target_num"], method="pearson")
corr_with_target = corr_onb[["target_num"]].sort_values("target_num", ascending=False)
print(corr_with_target)

plt.figure(figsize=(7, 10))
sns.heatmap(corr_with_target, annot=True)
#plt.title("Pearson correlation: onboarding scores vs target")
plt.tight_layout()
plt.savefig("../figures/correlation/correlation_pearson_onboarding_target.pdf")
plt.show()
corr_with_target.to_csv("../figures/correlation/correlation_pearson_onboarding_target.csv")
corr_with_target.to_latex("../figures/correlation/correlation_pearson_onboarding_target.tex", float_format="{:0.2f}".format)

### demographic correlation

In [ ]:
demo_cols = ["Alter:", "Körpergröße (in cm):", "Gewicht (in kg):"]
demo_cols = [c for c in demo_cols if c in dfm.columns]
eta_demo = eta_corr(dfm, "target_clean", demo_cols)
print(eta_demo)

sns.heatmap(eta_demo, annot=True)
#plt.title("correlation ratio: demographics")
plt.tight_layout()
plt.savefig("../figures/correlation/correlation_eta_demo.pdf")
plt.show()

In [ ]:
demo_cat = ["Geschlecht:"]
V_demo_target = cramers_v_corr(dfm, demo_cat + ["target_clean"])
sex_v = V_demo_target.loc["Geschlecht:", "target_clean"]

sns.heatmap([[sex_v]], annot=True, xticklabels=["target_clean"], yticklabels=["Geschlecht:"])
#plt.title("cramer's V correlation: sex")
plt.tight_layout()
plt.savefig("../figures/correlation/correlation_cramersV_sex_target.pdf")
plt.show()

## CO2 skewness

In [ ]:
dfm["CO2_log1p"] = np.log1p(dfm["CO2"].clip(lower=0))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(dfm["CO2"], kde=True, ax=axes[0])
axes[0].set_title("CO2 (raw)")

sns.histplot(dfm["CO2_log1p"], kde=True, ax=axes[1])
axes[1].set_title("CO2 (log1p-transformed)")
plt.tight_layout()
plt.show()

print("Skew (raw):", dfm["CO2"].skew())
print("Skew (log1p):", dfm["CO2_log1p"].skew())


## PCA + UMAP + tSNE

In [ ]:
# setting up the dataset
#DATA_PATH = data_dir()
#main_path = os.path.join(DATA_PATH,"all_aircontrol_with_surveys_hr_FIXED.csv")
#onb_path  = os.path.join(DATA_PATH, "onboarding_survey.csv")
#dfm = build_dfm(main_path, onb_path)

main_path = "../../data/personai/all_aircontrol_with_surveys_hr_with_new_bpm_new.csv"
onb_path = "../../data/personai/onboarding_survey.csv"
dfm = build_dfm(main_path, onb_path)
dfm["q_preference_temperature"] = dfm["q_preference_temperature"].map(LABEL_MAP)

# using corelation ratio (eta)
if not {"tod"}.issubset(dfm.columns):
    dt = pd.to_datetime(dfm["survey_time"], errors="coerce")
    dfm["tod"] = dt.dt.hour + dt.dt.minute/60 #converting every time to a numerical value(13:30 -> 13.5)

#deskewing CO2 with log1p
dfm["CO2_log1p"] = np.log1p(pd.to_numeric(dfm["CO2"], errors="coerce").clip(lower=0))

num_cols = dfm.select_dtypes(include=[float, int]).columns.tolist()
num_cols = [c for c in num_cols if c not in ["Unnamed: 0","CO2"]] 
if "CO2_log1p" not in num_cols:
    num_cols.append("CO2_log1p")

X = pd.concat([dfm[num_cols]], axis=1)
X = X.replace([np.inf, -np.inf], np.nan).dropna(axis=0)

y = dfm.loc[X.index, "q_preference_temperature"]
scaled = StandardScaler().fit_transform(X)

print("numerical features:")
print(X.columns.tolist())
print(X.columns.size)

all_cat_cols = ["q_location", "q_activity", "q_feeling", "q_heartbeat", "bpm_method", "Geschlecht:", "Höchster Bildungsabschluss:","Haben Sie mehrere Jahre (5+) außerhalb von Österreich gelebt?", 
            "Art der Wohnimmobilie:", "Baualter der Immobilie", "Klimatisierungsmöglichkeit:", "Heizungssystem:", "Wurde die Immobilie in den letzten 20 Jahren saniert?", 
            "Wie würden Sie das durchschnittliche Raumklima in Ihrem Zuhause einschätzen? Unter Raumklima verstehen wir Faktoren wie Temperatur, Luftfeuchtigkeit, Lichtverhältnisse, Lärmpegel, Luftfrische usw.", 
            "Berufliche Tätigkeit (vorwiegend):", "Geben Sie bitte die Räumlichkeiten in Ihrem Büro an:", "Klimatisierungsmöglichkeit im Büro:", "Heizungssystem im Büro:", 
            "Wie würden Sie das durchschnittliche Raumklima in Ihrem Büro einschätzen? Unter Raumklima verstehen wir Faktoren wie Temperatur, Luftfeuchtigkeit, Lichtverhältnisse, Lärmpegel, Luftfrische usw.", 
            "Wie schätzen Sie Ihr Fitnesslevel ein?"]

all_df_enc = one_hot_encode(dfm,all_cat_cols)

all_X = pd.concat([dfm[num_cols], all_df_enc], axis=1)
all_X = all_X.replace([np.inf, -np.inf], np.nan).dropna(axis=0)

all_y = dfm.loc[all_X.index, "q_preference_temperature"]
all_scaled = StandardScaler().fit_transform(all_X)

print("numerical + all categorical features:")
print(all_X.columns.tolist())
print(all_X.columns.size)


main_cat_cols = ["q_location", "q_activity", "q_feeling", "q_heartbeat", "Geschlecht:","Wie schätzen Sie Ihr Fitnesslevel ein?"]
main_df_enc = one_hot_encode(dfm,main_cat_cols)

main_X = pd.concat([dfm[num_cols], main_df_enc], axis=1)
main_X = main_X.replace([np.inf, -np.inf], np.nan).dropna(axis=0)

main_y = dfm.loc[main_X.index, "q_preference_temperature"]
main_scaled = StandardScaler().fit_transform(main_X)

print("numerical + main categorical features:")
print(main_X.columns.tolist())
print(main_X.columns.size)

### PCA

In [ ]:
# only numerical features PCA 2D
pca = PCA(n_components=2, random_state=42)
Z = pca.fit_transform(scaled)
print("Explained variance ratio (PC1, PC2):", pca.explained_variance_ratio_)

df_pca = pd.DataFrame(Z, columns=["PC1", "PC2"], index=X.index)
df_pca["target"] = y

sns.scatterplot(data=df_pca, x="PC1", y="PC2", palette=COLOR_MAP, hue="target")
#plt.title("PCA with numerical features only(2D)")
plt.tight_layout()
plt.savefig("../figures/pca/pca_numerical_2d.pdf")
plt.show()

In [ ]:
# all categorical features PCA 2D
pca = PCA(n_components=2, random_state=42)
Z = pca.fit_transform(all_scaled)
print("Explained variance ratio (PC1, PC2):", pca.explained_variance_ratio_)

df_pca = pd.DataFrame(Z, columns=["PC1", "PC2"], index=all_X.index)
df_pca["target"] = all_y

sns.scatterplot(data=df_pca, x="PC1", y="PC2", palette=COLOR_MAP, hue="target")
#plt.title("PCA with numerical features + all categorical features(2D)")
plt.tight_layout()
plt.savefig("../figures/pca/pca_numerical_all_categorical_2d.pdf")
plt.show()

In [ ]:
# main categorical features PCA 2D
pca = PCA(n_components=2, random_state=42)
Z = pca.fit_transform(main_scaled)
print("Explained variance ratio (PC1, PC2):", pca.explained_variance_ratio_)

df_pca = pd.DataFrame(Z, columns=["PC1", "PC2"], index=main_X.index)
df_pca["target"] = main_y

sns.scatterplot(data=df_pca, x="PC1", y="PC2", palette=COLOR_MAP, hue="target")
#plt.title("PCA with numerical features + main categorical features(2D)")
plt.tight_layout()
plt.savefig("../figures/pca/pca_numerical_main_categorical_2d.pdf")
plt.show()

In [ ]:
# only numerical features PCA 3D
pca3 = PCA(n_components=3, random_state=42)
Z3 = pca3.fit_transform(scaled)
print("Explained variance ratio (PC1, PC2, PC3):", pca3.explained_variance_ratio_)

df_pca3 = pd.DataFrame(Z3, columns=["PC1", "PC2", "PC3"], index=X.index)
df_pca3["target"] = y

fig = plt.figure(figsize=(7,6), dpi=130)
ax = fig.add_subplot(111, projection="3d")

for label in df_pca3["target"].unique():
    mask = df_pca3["target"] == label
    ax.scatter(df_pca3.loc[mask, "PC1"],df_pca3.loc[mask, "PC2"],df_pca3.loc[mask, "PC3"],label=label, color=COLOR_MAP[label], alpha=0.5)

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
#ax.set_title("PCA (3D)")
ax.legend(title="target")
plt.tight_layout()
plt.savefig("../figures/pca/pca_numerical_3d.pdf")
plt.show()

In [ ]:
# all categorical features PCA 3D
pca3 = PCA(n_components=3, random_state=42)
Z3 = pca3.fit_transform(all_scaled)
print("Explained variance ratio (PC1, PC2, PC3):", pca3.explained_variance_ratio_)

df_pca3 = pd.DataFrame(Z3, columns=["PC1", "PC2", "PC3"], index=all_X.index)
df_pca3["target"] = all_y

fig = plt.figure(figsize=(7,6), dpi=130)
ax = fig.add_subplot(111, projection="3d")

for label in df_pca3["target"].unique():
    mask = df_pca3["target"] == label
    ax.scatter(df_pca3.loc[mask, "PC1"],df_pca3.loc[mask, "PC2"],df_pca3.loc[mask, "PC3"],label=label, color=COLOR_MAP[label], alpha=0.5)

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
#ax.set_title("PCA with numerical features + all categorical features(3D)")
ax.legend(title="target")
plt.tight_layout()
plt.savefig("../figures/pca/pca_numerical_all_categorical_3d.pdf")
plt.show()

In [ ]:
# main categorical features PCA 3D
pca3 = PCA(n_components=3, random_state=42)
Z3 = pca3.fit_transform(main_scaled)
print("Explained variance ratio (PC1, PC2, PC3):", pca3.explained_variance_ratio_)

df_pca3 = pd.DataFrame(Z3, columns=["PC1", "PC2", "PC3"], index=main_X.index)
df_pca3["target"] = main_y

fig = plt.figure(figsize=(7,6), dpi=130)
ax = fig.add_subplot(111, projection="3d")

for label in df_pca3["target"].unique():
    mask = df_pca3["target"] == label
    ax.scatter(df_pca3.loc[mask, "PC1"],df_pca3.loc[mask, "PC2"],df_pca3.loc[mask, "PC3"],label=label,color=COLOR_MAP[label], alpha=0.5)

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
#ax.set_title("PCA with numerical features + main categorical features(3D)")
ax.legend(title="target")
plt.tight_layout()
plt.savefig("../figures/pca/pca_numerical_main_categorical_3d.pdf")
plt.show()

In [ ]:
# only numerical features PCA 95% variance 
pca95 = PCA(n_components=0.95, random_state=42).fit(scaled)
Z95 = pca95.transform(scaled)
print("PCs kept for 95% variance:", pca95.n_components_)
print("Total explained variance:", pca95.explained_variance_ratio_.sum())

df_pca95 = pd.DataFrame(Z95[:, :2], columns=["PC1", "PC2"], index=X.index)
df_pca95["target"] = y

sns.scatterplot(data=df_pca95, x="PC1", y="PC2", palette=COLOR_MAP, hue="target")
plt.title("PCA with numerical features only(95%)")
plt.tight_layout()
plt.show()

In [ ]:
# all categorical features PCA 95% variance 
pca95 = PCA(n_components=0.95, random_state=42).fit(all_scaled)
Z95 = pca95.transform(all_scaled)
print("PCs kept for 95% variance:", pca95.n_components_)
print("Total explained variance:", pca95.explained_variance_ratio_.sum())

df_pca95 = pd.DataFrame(Z95[:, :2], columns=["PC1", "PC2"], index=all_X.index)
df_pca95["target"] = all_y

sns.scatterplot(data=df_pca95, x="PC1", y="PC2", palette=COLOR_MAP, hue="target")
plt.title("PCA with numerical features + all categorical features(95%)")
plt.tight_layout()
plt.show()

In [ ]:
# main categorical features PCA 95% variance 
pca95 = PCA(n_components=0.95, random_state=42).fit(main_scaled)
Z95 = pca95.transform(main_scaled)
print("PCs kept for 95% variance:", pca95.n_components_)
print("Total explained variance:", pca95.explained_variance_ratio_.sum())

df_pca95 = pd.DataFrame(Z95[:, :2], columns=["PC1", "PC2"], index=main_X.index)
df_pca95["target"] = main_y

sns.scatterplot(data=df_pca95, x="PC1", y="PC2", palette=COLOR_MAP, hue="target")
plt.title("PCA with numerical features + main categorical features(95%)")
plt.tight_layout()
plt.show()

### UMAP

In [ ]:
# only numerical features UMAP 2D
reducer = umap.UMAP(n_components=2, random_state=42)
Z_umap = reducer.fit_transform(scaled)

df_umap = pd.DataFrame(Z_umap, columns=["UMAP1", "UMAP2"], index=X.index)
df_umap["target"] = y

sns.scatterplot(data=df_umap, x="UMAP1", y="UMAP2", palette=COLOR_MAP, hue="target")
plt.title("UMAP with numerical features only(2D)")
plt.tight_layout()
plt.show()

In [ ]:
# all categorical features UMAP 2D
reducer = umap.UMAP(n_components=2, random_state=42)
Z_umap = reducer.fit_transform(all_scaled)

df_umap = pd.DataFrame(Z_umap, columns=["UMAP1", "UMAP2"], index=all_X.index)
df_umap["target"] = all_y

sns.scatterplot(data=df_umap, x="UMAP1", y="UMAP2", palette=COLOR_MAP, hue="target")
plt.title("UMAP with numerical features + all categorical features(2D)")
plt.tight_layout()
plt.show()

In [ ]:
# main categorical features UMAP 2D
reducer = umap.UMAP(n_components=2, random_state=42)
Z_umap = reducer.fit_transform(main_scaled)

df_umap = pd.DataFrame(Z_umap, columns=["UMAP1", "UMAP2"], index=main_X.index)
df_umap["target"] = main_y

sns.scatterplot(data=df_umap, x="UMAP1", y="UMAP2", palette=COLOR_MAP, hue="target")
plt.title("UMAP with numerical features + main categorical features(2D)")
plt.tight_layout()
plt.show()

In [ ]:
# only numerical features UMAP 3D
reducer = umap.UMAP(n_components=3, random_state=42)
Z_umap = reducer.fit_transform(scaled)

df_umap3 = pd.DataFrame(Z_umap, columns=["UMAP1", "UMAP2", "UMAP3"], index=X.index)
df_umap3["target"] = y

fig = plt.figure(figsize=(7,6), dpi=130)
ax = fig.add_subplot(111, projection="3d")

for label in df_umap3["target"].unique():
    mask = df_umap3["target"] == label
    ax.scatter(df_umap3.loc[mask, "UMAP1"], df_umap3.loc[mask, "UMAP2"], df_umap3.loc[mask, "UMAP3"], label=label, color=COLOR_MAP[label], alpha=0.5)

ax.set_xlabel("UMAP1")
ax.set_ylabel("UMAP2")
ax.set_zlabel("UMAP3")
ax.set_title("UMAP with numerical features only(3D)")
ax.legend(title="target")
plt.tight_layout()
plt.show()

In [ ]:
# all categorical features UMAP 3D
reducer = umap.UMAP(n_components=3, random_state=42)
Z_umap = reducer.fit_transform(all_scaled)

df_umap3 = pd.DataFrame(Z_umap, columns=["UMAP1", "UMAP2", "UMAP3"], index=all_X.index)
df_umap3["target"] = all_y

fig = plt.figure(figsize=(7,6), dpi=130)
ax = fig.add_subplot(111, projection="3d")

for label in df_umap3["target"].unique():
    mask = df_umap3["target"] == label
    ax.scatter(df_umap3.loc[mask, "UMAP1"], df_umap3.loc[mask, "UMAP2"], df_umap3.loc[mask, "UMAP3"], label=label,color=COLOR_MAP[label], alpha=0.5)

ax.set_xlabel("UMAP1")
ax.set_ylabel("UMAP2")
ax.set_zlabel("UMAP3")
ax.set_title("UMAP with numerical features + all categorical features(3D)")
ax.legend(title="target")
plt.tight_layout()
plt.show()

In [ ]:
# main categorical features UMAP 3D
reducer = umap.UMAP(n_components=3, random_state=42)
Z_umap = reducer.fit_transform(main_scaled)

df_umap3 = pd.DataFrame(Z_umap, columns=["UMAP1", "UMAP2", "UMAP3"], index=main_X.index)
df_umap3["target"] = main_y

fig = plt.figure(figsize=(7,6), dpi=130)
ax = fig.add_subplot(111, projection="3d")

for label in df_umap3["target"].unique():
    mask = df_umap3["target"] == label
    ax.scatter(df_umap3.loc[mask, "UMAP1"], df_umap3.loc[mask, "UMAP2"], df_umap3.loc[mask, "UMAP3"], label=label, color=COLOR_MAP[label], alpha=0.5)

ax.set_xlabel("UMAP1")
ax.set_ylabel("UMAP2")
ax.set_zlabel("UMAP3")
ax.set_title("UMAP with numerical features + main categorical features(3D)")
ax.legend(title="target")
plt.tight_layout()
plt.show()

### tSNE

In [ ]:
# only numerical features t-SNE 2D
tsne = TSNE(n_components=2, random_state=42)
Z_tsne = tsne.fit_transform(scaled)

df_tsne = pd.DataFrame(Z_tsne, columns=["tSNE1", "tSNE2"], index=X.index)
df_tsne["target"] = y

sns.scatterplot(data=df_tsne, x="tSNE1", y="tSNE2", palette=COLOR_MAP, hue="target", s=25, alpha=0.9, edgecolor="none")
plt.title("t-SNE with numerical features only(2D)")
plt.tight_layout()
plt.show()

In [ ]:
# all categorical features t-SNE 2D
tsne = TSNE(n_components=2, random_state=42)
Z_tsne = tsne.fit_transform(all_scaled)

df_tsne = pd.DataFrame(Z_tsne, columns=["tSNE1", "tSNE2"], index=all_X.index)
df_tsne["target"] = all_y

sns.scatterplot(data=df_tsne, x="tSNE1", y="tSNE2", palette=COLOR_MAP, hue="target", s=25, alpha=0.9, edgecolor="none")
plt.title("t-SNE with numerical features + all categorical features(2D)")
plt.tight_layout()
plt.show()

In [ ]:
# main categorical features t-SNE 2D
tsne = TSNE(n_components=2, random_state=42)
Z_tsne = tsne.fit_transform(main_scaled)

df_tsne = pd.DataFrame(Z_tsne, columns=["tSNE1", "tSNE2"], index=main_X.index)
df_tsne["target"] = main_y

sns.scatterplot(data=df_tsne, x="tSNE1", y="tSNE2", palette=COLOR_MAP, hue="target", s=25, alpha=0.9, edgecolor="none")
plt.title("t-SNE with numerical features + main categorical features(2D)")
plt.tight_layout()
plt.show()

In [ ]:
# only numerical features t-SNE 3D
tsne3 = TSNE(n_components=3,random_state=42)
Z_tsne3 = tsne3.fit_transform(scaled)

df_tsne3 = pd.DataFrame(Z_tsne3, columns=["tSNE1", "tSNE2", "tSNE3"], index=X.index)
df_tsne3["target"] = y

fig = plt.figure(figsize=(7,6), dpi=130)
ax = fig.add_subplot(111, projection="3d")

for label in df_tsne3["target"].unique():
    m = df_tsne3["target"] == label
    ax.scatter(df_tsne3.loc[m,"tSNE1"], df_tsne3.loc[m,"tSNE2"], df_tsne3.loc[m,"tSNE3"], label=label, color=COLOR_MAP[label], alpha=0.5)

ax.set_xlabel("tSNE1")
ax.set_ylabel("tSNE2")
ax.set_zlabel("tSNE3")
ax.set_title("t-SNE with numerical features only(3D)")
ax.legend(title="target")
plt.tight_layout()
plt.show()

In [ ]:
# all categorical features t-SNE 3D
tsne3 = TSNE(n_components=3,random_state=42)
Z_tsne3 = tsne3.fit_transform(all_scaled)

df_tsne3 = pd.DataFrame(Z_tsne3, columns=["tSNE1", "tSNE2", "tSNE3"], index=all_X.index)
df_tsne3["target"] = all_y

fig = plt.figure(figsize=(7,6), dpi=130)
ax = fig.add_subplot(111, projection="3d")

for label in df_tsne3["target"].unique():
    m = df_tsne3["target"] == label
    ax.scatter(df_tsne3.loc[m,"tSNE1"], df_tsne3.loc[m,"tSNE2"], df_tsne3.loc[m,"tSNE3"], label=label, color=COLOR_MAP[label], alpha=0.5)

ax.set_xlabel("tSNE1")
ax.set_ylabel("tSNE2")
ax.set_zlabel("tSNE3")
ax.set_title("t-SNE with numerical features + all categorical features(3D)")
ax.legend(title="target")
plt.tight_layout()
plt.show()

In [ ]:
# main categorical features t-SNE 3D
tsne3 = TSNE(n_components=3,random_state=42)
Z_tsne3 = tsne3.fit_transform(main_scaled)

df_tsne3 = pd.DataFrame(Z_tsne3, columns=["tSNE1", "tSNE2", "tSNE3"], index=main_X.index)
df_tsne3["target"] = main_y

fig = plt.figure(figsize=(7,6), dpi=130)
ax = fig.add_subplot(111, projection="3d")

for label in df_tsne3["target"].unique():
    m = df_tsne3["target"] == label
    ax.scatter(df_tsne3.loc[m,"tSNE1"], df_tsne3.loc[m,"tSNE2"], df_tsne3.loc[m,"tSNE3"], label=label, color=COLOR_MAP[label], alpha=0.5)

ax.set_xlabel("tSNE1")
ax.set_ylabel("tSNE2")
ax.set_zlabel("tSNE3")
ax.set_title("t-SNE with numerical features + main categorical features(3D)")
ax.legend(title="target")
plt.tight_layout()
plt.show()

### pairplot with thermal prefernces

In [ ]:
#order = [ "Keine Veränderung", "Kühler","Wärmer"]
order = [ "no change", "prefer cooler","prefer warmer"]
sns.pairplot(dfm.loc[:, ['AirTemp', 'Humidity', 'CO2', 'bpm', 'CO2_log1p', 'q_preference_temperature']], palette=COLOR_MAP, hue='q_preference_temperature', hue_order=order)

### PCA per user

In [ ]:
# using corelation ratio (eta)
if not {"tod"}.issubset(dfm.columns):
    dt = pd.to_datetime(dfm["survey_time"], errors="coerce")
    dfm["tod"] = dt.dt.hour + dt.dt.minute/60 #converting every time to a numerical value(13:30 -> 13.5)

#deskewing CO2 with log1p
dfm["CO2_log1p"] = np.log1p(pd.to_numeric(dfm["CO2"], errors="coerce").clip(lower=0))

num_cols = dfm.select_dtypes(include=[float, int]).columns.tolist()
num_cols = [c for c in num_cols if c not in ["Unnamed: 0","CO2"]] 
if "CO2_log1p" not in num_cols:
    num_cols.append("CO2_log1p")

all_cat_cols = ["q_location", "q_activity", "q_feeling", "q_heartbeat", "bpm_method", "Geschlecht:", "Höchster Bildungsabschluss:","Haben Sie mehrere Jahre (5+) außerhalb von Österreich gelebt?", 
            "Art der Wohnimmobilie:", "Baualter der Immobilie", "Klimatisierungsmöglichkeit:", "Heizungssystem:", "Wurde die Immobilie in den letzten 20 Jahren saniert?", 
            "Wie würden Sie das durchschnittliche Raumklima in Ihrem Zuhause einschätzen? Unter Raumklima verstehen wir Faktoren wie Temperatur, Luftfeuchtigkeit, Lichtverhältnisse, Lärmpegel, Luftfrische usw.", 
            "Berufliche Tätigkeit (vorwiegend):", "Geben Sie bitte die Räumlichkeiten in Ihrem Büro an:", "Klimatisierungsmöglichkeit im Büro:", "Heizungssystem im Büro:", 
            "Wie würden Sie das durchschnittliche Raumklima in Ihrem Büro einschätzen? Unter Raumklima verstehen wir Faktoren wie Temperatur, Luftfeuchtigkeit, Lichtverhältnisse, Lärmpegel, Luftfrische usw.", 
            "Wie schätzen Sie Ihr Fitnesslevel ein?"]

all_df_enc = one_hot_encode(dfm,all_cat_cols)

all_X = pd.concat([dfm[num_cols], all_df_enc], axis=1)
all_X = all_X.replace([np.inf, -np.inf], np.nan).dropna(axis=0)

all_y = dfm.loc[all_X.index, "q_preference_temperature"]
all_scaled = StandardScaler().fit_transform(all_X)

In [ ]:
from sklearn.decomposition import PCA
import numpy as np
import matplotlib.pyplot as plt

TARGET_COL = "q_preference_temperature"
USER_COL   = "user_id"
MIN_ROWS, N_SAMPLE = 50, 12

users = dfm.loc[all_X.index, USER_COL].values
y = all_y.values

#global PCA fit (using your all_scaled matrix)
pca = PCA(n_components=2, random_state=0)
PC = pca.fit_transform(all_scaled)
print(f"Explained variance: PC1={pca.explained_variance_ratio_[0]:.2%}, PC2={pca.explained_variance_ratio_[1]:.2%}")

#build dataframe for easy grouping
df_pc = pd.DataFrame({"PC1": PC[:, 0], "PC2": PC[:, 1], TARGET_COL: y, USER_COL: users})

#pick valid users (≥MIN_ROWS and ≥2 target classes)
valid_users = [u for u, d in df_pc.groupby(USER_COL) if len(d) >= MIN_ROWS and d[TARGET_COL].nunique() > 1]

#randomly sample a few users
np.random.seed(0)
sample_users = np.random.choice(valid_users, size=min(N_SAMPLE, len(valid_users)), replace=False)
print(f"Plotting {len(sample_users)} users")

#plots
for uid in sample_users:
    d = df_pc[df_pc[USER_COL] == uid]
    plt.figure(figsize=(5, 4))
    for lab, sub in d.groupby(TARGET_COL):
        plt.scatter(sub["PC1"], sub["PC2"], s=25, alpha=0.7, label=str(lab), color=COLOR_MAP[lab])
    plt.title(f"PCA — user {uid}")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.legend(title="Preference")
    plt.tight_layout()
    plt.show()

### Similarity Analysis 

In [ ]:
import re, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import sys, pathlib
#sys.path.append("..")
#from anchor import data_dir


def build_dfm(main_path, onb_path,id_main="user_id", id_onb="id",target_col="q_preference_temperature"):
    df  = pd.read_csv(main_path)
    onb = pd.read_csv(onb_path)
    df.columns  = df.columns.str.replace(r"\s+", " ", regex=True).str.strip()
    onb.columns = onb.columns.str.replace(r"\s+", " ", regex=True).str.strip()
    df[id_main]  = df[id_main].astype(str).str.strip()
    onb[id_onb]  = onb[id_onb].astype(str).str.strip()
    onb = onb.sort_values(id_onb).drop_duplicates(subset=[id_onb], keep="first")
    dfm = df.merge(onb, left_on=id_main, right_on=id_onb, how="left")

    def clean_de(s):
        s = str(s).lower()
        s = unicodedata.normalize("NFKD", s).encode("ascii","ignore").decode("utf-8")  # kühler->kuhler
        s = re.sub(r"[-_]+"," ", s)
        s = re.sub(r"\s+"," ", s).strip()
        return s
    
    #dfm["target_clean"] = dfm[target_col].map(clean_de).replace({"keineveranderung":"keine verand erung"})
    return dfm

#DATA_PATH = data_dir()
#main_path = os.path.join(DATA_PATH,"all_aircontrol_with_surveys_hr_FIXED.csv")
#onb_path  = os.path.join(DATA_PATH, "onboarding_survey.csv")
#dfm = build_dfm(main_path, onb_path)

main_path = "../../data/personai/all_aircontrol_with_surveys_hr_with_new_bpm_new.csv"
onb_path = "../../data/personai/onboarding_survey.csv"
dfm = build_dfm(main_path, onb_path)
dfm["q_preference_temperature"] = dfm["q_preference_temperature"].map(LABEL_MAP)    

if "CO2_log1p" not in dfm.columns:
    dfm["CO2_log1p"] = np.log1p(pd.to_numeric(dfm["CO2"], errors="coerce").clip(lower=0))
if "CO2" in dfm.columns:
    dfm = dfm.drop(columns=["CO2"])

In [ ]:
TARGET_COL = "q_preference_temperature"
NO_CHANGE, COOLER, WARMER = "no change", "prefer cooler", "prefer warmer"

#preparing numeical and categorical features
num_cols = [c for c in dfm.columns if c != TARGET_COL and pd.api.types.is_numeric_dtype(dfm[c])]
cat_cols = ["q_location", "q_activity", "q_feeling", "q_heartbeat", "bpm", "Geschlecht:", "Höchster Bildungsabschluss:","Haben Sie mehrere Jahre (5+) außerhalb von Österreich gelebt?", 
            "Art der Wohnimmobilie:", "Baualter der Immobilie", "Klimatisierungsmöglichkeit:", "Heizungssystem:", "Wurde die Immobilie in den letzten 20 Jahren saniert?", 
            "Wie würden Sie das durchschnittliche Raumklima in Ihrem Zuhause einschätzen? Unter Raumklima verstehen wir Faktoren wie Temperatur, Luftfeuchtigkeit, Lichtverhältnisse, Lärmpegel, Luftfrische usw.", 
            "Berufliche Tätigkeit (vorwiegend):", "Geben Sie bitte die Räumlichkeiten in Ihrem Büro an:", "Klimatisierungsmöglichkeit im Büro:", "Heizungssystem im Büro:", 
            "Wie würden Sie das durchschnittliche Raumklima in Ihrem Büro einschätzen? Unter Raumklima verstehen wir Faktoren wie Temperatur, Luftfeuchtigkeit, Lichtverhältnisse, Lärmpegel, Luftfrische usw.", 
            "Wie schätzen Sie Ihr Fitnesslevel ein?"] #bpm_method

#encoding categorical features as one-hot and preparing final X and y
X_num = dfm[num_cols].replace([np.inf, -np.inf], np.nan)
X_cat = pd.get_dummies(dfm[cat_cols].fillna("Missing"), drop_first=False, dtype=np.uint8)
y = dfm[TARGET_COL]

#dropping rows with NaN/inf in numeric features
before_rows = len(X_num)
X_num = X_num.dropna(axis=0)
X_cat = X_cat.loc[X_num.index]
y = y.loc[X_num.index]
after_rows = len(X_num)
print(f"Number of rows before dropping NaN/inf: {before_rows}, after: {after_rows}, dropped: {before_rows - after_rows}")


#scaling numerical features and combining with categorical(one-hot)features
X_num_scaled = pd.DataFrame(StandardScaler().fit_transform(X_num), index=X_num.index, columns=X_num.columns)
X = pd.concat([X_num_scaled, X_cat], axis=1).values

#splitting indices by target class
yv = y.values
idx_nc = np.where(yv == NO_CHANGE)[0]
idx_c  = np.where(yv == COOLER)[0]
idx_w  = np.where(yv == WARMER)[0]
X_nc, X_c_, X_w_ = X[idx_nc], X[idx_c], X[idx_w]

#safe cosine similarity function to handle empty arrays
def safe_cos(A, B):
    if (A.size==0 or B.size==0):
        return np.empty((A.shape[0], 0))
    return cosine_similarity(A, B)

#computing similarities
S_nc_c, S_nc_w = safe_cos(X_nc, X_c_), safe_cos(X_nc, X_w_)
if S_nc_c.shape[1]:
    mean_c = S_nc_c.mean(axis=1)
    max_c = S_nc_c.max(axis=1)
else:
    mean_c = np.full(len(X_nc), np.nan)
    max_c = np.full(len(X_nc), np.nan)

if S_nc_w.shape[1]:
    mean_w = S_nc_w.mean(axis=1)
    max_w = S_nc_w.max(axis=1)
else:
    mean_w = np.full(len(X_nc), np.nan)
    max_w = np.full(len(X_nc), np.nan)


summary = pd.DataFrame({
    "mean_sim_to_Cooler": mean_c,
    "mean_sim_to_Warmer": mean_w,
    "max_sim_to_Cooler": max_c,
    "max_sim_to_Warmer": max_w,
})
summary["closer_class_by_mean"] = np.where(summary["mean_sim_to_Cooler"] > summary["mean_sim_to_Warmer"], COOLER, WARMER)

#function to compute percentage of values >= threshold
def above_thr(s, t):
    s = s.dropna()
    if len(s):
        return (s >= t).mean() * 100
    return np.nan

for thr in (0.8, 0.9):
    print(f"≥{thr} mean sim — Cooler: {above_thr(summary.mean_sim_to_Cooler, thr)}% | Warmer: {above_thr(summary.mean_sim_to_Warmer, thr)}%")

print("\nNoChange closer-to (%):")
print((summary["closer_class_by_mean"].value_counts(normalize=True)*100).round(1))

#plots
plt.figure(figsize=(6,4))
plt.hist(summary["mean_sim_to_Cooler"].dropna(), bins=30)
plt.title("NoChange → Cooler: mean cosine similarity")
plt.xlabel("similarity")
plt.ylabel("count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4))
plt.hist(summary["mean_sim_to_Warmer"].dropna(), bins=30)
plt.title("NoChange → Warmer: mean cosine similarity")
plt.xlabel("similarity")
plt.ylabel("count")
plt.tight_layout()
plt.show()

#for each entery in no change catefgory, take the best similarity among cooler and warmer
plt.figure(figsize=(6,4))
best = np.nanmax(np.c_[summary["mean_sim_to_Cooler"], summary["mean_sim_to_Warmer"]], axis=1)
best_no_nan = best[np.isnan(best) == False]
plt.hist(best_no_nan, bins=30)
plt.title("NoChange: best-of-two mean similarity")
plt.xlabel("similarity")
plt.ylabel("count")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import pandas as pd

#clustering the no-change samples based on their mean similarities
kmeans = KMeans(random_state=0)
summary["nochange_cluster"] = kmeans.fit_predict(summary[["mean_sim_to_Cooler", "mean_sim_to_Warmer"]])

print("Cluster counts:")
print(summary["nochange_cluster"].value_counts())

#plots
plt.figure(figsize=(6,5))
scatter = plt.scatter(
    summary["mean_sim_to_Cooler"],
    summary["mean_sim_to_Warmer"],
    c=summary["nochange_cluster"],
    cmap="viridis", alpha=0.7, s=40, edgecolor="k"
)

#cluster centers
centers = kmeans.cluster_centers_
plt.scatter(centers[:,0], centers[:,1], label="Cluster centers")

plt.xlabel("Mean similarity to Cooler")
plt.ylabel("Mean similarity to Warmer")
plt.title("Subgroups within 'No change' samples (by mean similarities)")
plt.legend()
plt.tight_layout()
plt.show()